# COVID-19 Wikipedia Table: Scrape and Clean

One of the first things I wanted to know when I started learning pandas was whether I could take a messy, real-world HTML table and turn it into something I could actually analyze. This notebook pulls the country-level COVID-19 statistics table straight off Wikipedia, cleans it up, and asks a simple question: for countries with a high case count relative to deaths, versus countries with a low ratio, what does that ratio actually tell us?

No modeling here, just scraping and wrangling. This was my first real exposure to the fact that most of a data project is spent getting the data into a usable shape before any analysis can start.


In [ ]:
import pandas as pd
import bs4 as bs
import numpy as np
import requests
import re


## Step 1: Pull the table

The target is the country-by-country cases/deaths table on the Wikipedia page for the COVID-19 pandemic. I used `requests` to fetch the raw page and `BeautifulSoup` to isolate the specific table by its HTML id.


In [ ]:
source = requests.get("https://en.wikipedia.org/wiki/COVID-19_pandemic_by_country_and_territory")
print(source)


## Step 2: Parse it into a DataFrame

`pandas.read_html` can parse an HTML table directly once BeautifulSoup has located it. I kept only the four columns I actually needed: country, deaths per million, total deaths, and total cases.

Note: this notebook hardcodes the Wikipedia table id (`table65150380`), which is fragile. Wikipedia regenerates these ids whenever the page structure changes, so this cell will likely need updating if run again today. That is a known limitation, not an oversight.


In [ ]:
soup = bs.BeautifulSoup(source.content, features='html.parser')

covid_df = pd.read_html(str(soup.find('table', attrs = {'id':'table65150380'})))[0]
print(covid_df.columns)
covid_df = covid_df.loc[:,['Country.1', 'Deathsper million', 'Deaths', 'Cases']]
print(covid_df.head())


## Step 3: Rename columns for clarity


In [ ]:
covid_df.columns = ['country', 'deaths_per_million', 'deaths', 'cases']
covid_df.head(3)


## Step 4: Drop rows that are not usable

Wikipedia's table includes an aggregate summary row that is not a country and rows with no recorded deaths (represented as an em dash rather than a number). Both need to go before the data can be cast to numeric types.

The row index dropped below (`217`) was the aggregate/world-total row at the time this table was scraped. Hardcoding a row index by position is brittle if the table changes, but it does the job for this one-off exercise.


In [ ]:
# Row 217 was the non-country aggregate row in this version of the table
covid_df = covid_df.drop([217], axis = 0)
print(covid_df.head())
covid_df = covid_df.replace('—', '0')
covid_df = covid_df[covid_df.deaths != '0']


## Step 5: Clean up country names

Some country names carry footnote markers like `World[a]`. A regex strips anything in square brackets.


In [ ]:
covid_df.country = covid_df['country'].replace("\[.*\]", "",regex=True)
print(covid_df.head())


## Step 6: Index by country, cast types


In [ ]:
covid_df = covid_df.set_index('country')


In [ ]:
covid_df = covid_df.astype({'deaths_per_million':int, 'deaths':int, 'cases':int})


## Step 7: Derive a cases-per-death ratio

This is the number I actually cared about: for every recorded death, how many confirmed cases were there. A high ratio suggests either strong healthcare outcomes, younger demographics, or under-reporting of deaths relative to cases. A low ratio suggests the opposite, or under-testing that is inflating the apparent severity.


In [ ]:
covid_df['cases_per_deaths'] = round(covid_df['cases']/covid_df['deaths'])
print(covid_df.head())


## Step 8: Sort and inspect


In [ ]:
covid_df.sort_values('cases_per_deaths', ascending = False).head(20)


## What the ratio actually tells us

The `cases_per_deaths` column is a rough proxy for case fatality rate, inverted. Countries at the high end of the ranking show something on the order of two thousand cases for every one recorded death, while countries at the low end show closer to five cases per death.

I want to be upfront about what this comparison does and does not show. This is a single static snapshot, not a time series, so it does not account for reporting lag, testing capacity differences between countries, population age structure, or healthcare system capacity, all of which materially affect this ratio. Reading this as a direct measure of "how well a country handled COVID" would be a mistake. It is descriptive, not causal.

**Possible follow-up (not done here):** normalize by testing rate per capita and stratify by median age to see how much of the spread in this ratio survives after controlling for those two confounders.
